In [1]:
from pathlib import Path
import json
from ultralytics import YOLO
import random
from collections import defaultdict
from dataclasses import dataclass
from typing import Dict, List, Set, Tuple

from annotation_methods.budget_splits import make_nested_splits, Params
from annotation_methods.yolo_helpers import convert_all_inst_splits_to_yolo, write_yolo_dataset_yaml
from annotation_methods.io_utils import write_coco_output

In [8]:
make_nested_splits(
    train_json=Path("~/Thesis_WDV/Data/apples/annotations/instances_train.json").expanduser(),
    out_root=Path("~/Thesis_WDV/Data/apples/yolo_splits").expanduser(),
    params=Params(budgets=(250, 500, 1000), val_frac=0.2, seed=42),
)
make_nested_splits(
    train_json=Path("~/Thesis_WDV/Data/tomatoes/annotations/instances_train.json").expanduser(),
    out_root=Path("~/Thesis_WDV/Data/tomatoes/yolo_splits").expanduser(),
    params=Params(budgets=(250, 500, 1000), val_frac=0.2, seed=42),
)

Wrote splits to: /home/warredv/Thesis_WDV/Data/apples/yolo_splits
Wrote splits to: /home/warredv/Thesis_WDV/Data/tomatoes/yolo_splits


In [9]:
# run MANY inst splits

insts = [250,500,1000]
all_results = convert_all_inst_splits_to_yolo(
    inst_values=insts,
    coco_json="~/Thesis_WDV/Data/apples/annotations/instances_train.json",
    dataset_root="~/Thesis_WDV/Data/apples",
    splits_root="~/Thesis_WDV/Data/apples/yolo_splits",
)
for r in all_results:
    print(f"inst{r['inst']} TRAIN:", r["train"])
    print(f"inst{r['inst']} VAL  :", r["val"])

insts = [250,500,1000]
all_results = convert_all_inst_splits_to_yolo(
    inst_values=insts,
    coco_json="~/Thesis_WDV/Data/tomatoes/annotations/instances_train.json",
    dataset_root="~/Thesis_WDV/Data/tomatoes",
    splits_root="~/Thesis_WDV/Data/tomatoes/yolo_splits",
)
for r in all_results:
    print(f"inst{r['inst']} TRAIN:", r["train"])
    print(f"inst{r['inst']} VAL  :", r["val"])

inst250 TRAIN: {'processed_images': 16, 'labels_written': 208, 'missing_images_on_disk': 0, 'missing_manifest_entries_in_coco': 0, 'nc': 2, 'class_names': ['GoodApple', 'BadApple']}
inst250 VAL  : {'processed_images': 3, 'labels_written': 52, 'missing_images_on_disk': 0, 'missing_manifest_entries_in_coco': 0, 'nc': 2, 'class_names': ['GoodApple', 'BadApple']}
inst500 TRAIN: {'processed_images': 29, 'labels_written': 398, 'missing_images_on_disk': 0, 'missing_manifest_entries_in_coco': 0, 'nc': 2, 'class_names': ['GoodApple', 'BadApple']}
inst500 VAL  : {'processed_images': 8, 'labels_written': 108, 'missing_images_on_disk': 0, 'missing_manifest_entries_in_coco': 0, 'nc': 2, 'class_names': ['GoodApple', 'BadApple']}
inst1000 TRAIN: {'processed_images': 57, 'labels_written': 792, 'missing_images_on_disk': 0, 'missing_manifest_entries_in_coco': 0, 'nc': 2, 'class_names': ['GoodApple', 'BadApple']}
inst1000 VAL  : {'processed_images': 16, 'labels_written': 209, 'missing_images_on_disk': 0,

In [10]:
# Example:
write_yolo_dataset_yaml("~/Thesis_WDV/Data/apples/yolo_splits/inst250")
write_yolo_dataset_yaml("~/Thesis_WDV/Data/apples/yolo_splits/inst500")
write_yolo_dataset_yaml("~/Thesis_WDV/Data/apples/yolo_splits/inst1000")

write_yolo_dataset_yaml("~/Thesis_WDV/Data/tomatoes/yolo_splits/inst250")
write_yolo_dataset_yaml("~/Thesis_WDV/Data/tomatoes/yolo_splits/inst500")
write_yolo_dataset_yaml("~/Thesis_WDV/Data/tomatoes/yolo_splits/inst1000")


PosixPath('/home/warredv/Thesis_WDV/Data/tomatoes/yolo_splits/inst1000/dataset.yaml')

In [12]:
# Load a model
model = YOLO("yolo11n.pt")

train_results = model.train(
    data="../Data/apples/yolo_splits/inst500/dataset.yaml",
    epochs=100,
    imgsz=640,
    device="cuda",
    project="runs/apples",     # base directory
    name="yolo11n_inst500",    # custom experiment name
)


New https://pypi.org/project/ultralytics/8.3.241 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.239 🚀 Python-3.11.14 torch-2.9.1 CUDA:0 (NVIDIA A10G, 22588MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../Data/apples/yolo_splits/inst500/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolo11n_inst500, nbs=6

 10                  -1  1    249728  ultralytics.nn.modules.block.C2PSA           [256, 256, 1]                 
 11                  -1  1         0  torch.nn.modules.upsampling.Upsample         [None, 2, 'nearest']          
 12             [-1, 6]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 13                  -1  1    111296  ultralytics.nn.modules.block.C3k2            [384, 128, 1, False]          
 14                  -1  1         0  torch.nn.modules.upsampling.Upsample         [None, 2, 'nearest']          
 15             [-1, 4]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 16                  -1  1     32096  ultralytics.nn.modules.block.C3k2            [256, 64, 1, False]           
 17                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]                
 18            [-1, 13]  1         0  ultralytics.nn.modules.conv.Concat           [1]  

In [16]:
model = YOLO("runs/apples/yolo11n_inst500/weights/best.pt")

source = str(Path("~/Thesis_WDV/Data/apples/images/test").expanduser())
outdir = str(Path("~/Thesis_WDV/Yolo/yolo_outputs").expanduser())

model.predict(
    source=source,
    save=False,
    save_txt=True,
    save_conf=True,
    project=outdir,
    name="apples_test_yolo11n_inst500",
)

image 1/31 /home/warredv/Thesis_WDV/Data/apples/images/test/IMG_0298.png: 544x640 12 GoodApples, 8 BadApples, 7.7ms
image 2/31 /home/warredv/Thesis_WDV/Data/apples/images/test/IMG_0302.png: 544x640 12 GoodApples, 8 BadApples, 7.2ms
image 3/31 /home/warredv/Thesis_WDV/Data/apples/images/test/IMG_0307.png: 544x640 12 GoodApples, 8 BadApples, 7.3ms
image 4/31 /home/warredv/Thesis_WDV/Data/apples/images/test/IMG_0326.png: 544x640 12 GoodApples, 7.2ms
image 5/31 /home/warredv/Thesis_WDV/Data/apples/images/test/IMG_0335.png: 544x640 12 GoodApples, 7.2ms
image 6/31 /home/warredv/Thesis_WDV/Data/apples/images/test/IMG_0337.png: 544x640 12 GoodApples, 7.2ms
image 7/31 /home/warredv/Thesis_WDV/Data/apples/images/test/IMG_0345.png: 544x640 12 GoodApples, 7.3ms
image 8/31 /home/warredv/Thesis_WDV/Data/apples/images/test/IMG_0359.png: 544x640 12 GoodApples, 7.2ms
image 9/31 /home/warredv/Thesis_WDV/Data/apples/images/test/IMG_0360.png: 544x640 12 GoodApples, 7.2ms
image 10/31 /home/warredv/Thesis_W

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'GoodApple', 1: 'BadApple'}
 obb: None
 orig_img: array([[[ 78,  95,  98],
         [ 69,  86,  90],
         [ 72,  88,  90],
         ...,
         [154, 172, 172],
         [196, 213, 212],
         [207, 221, 220]],
 
        [[ 60,  77,  80],
         [ 53,  70,  73],
         [ 53,  69,  71],
         ...,
         [131, 152, 153],
         [156, 174, 176],
         [151, 165, 167]],
 
        [[ 61,  77,  80],
         [ 58,  75,  78],
         [ 55,  71,  73],
         ...,
         [115, 136, 138],
         [118, 135, 138],
         [112, 126, 128]],
 
        ...,
 
        [[ 68,  77,  74],
         [ 87,  97,  95],
         [100, 111, 109],
         ...,
         [  3,   9,   8],
         [  5,  10,   9],
         [  7,   9,   9]],
 
        [[ 92, 100,  98],
         [105, 116, 114],
         [120, 131, 130],
         ...,


In [19]:
def yolo_pred_txt_to_coco_results(
    labels_dir,
    images_dir,
    output_path,
    model_name,
    categories_list,
    is_xywh_normalized=True,
    has_conf=True,
    total_train_time=0.0,
    total_inf_time=0.0,
):
    """
    Convert YOLO prediction .txt files into a COCO-format JSON file written via
    `write_coco_output()`.

    YOLO txt format assumed: cls xc yc w h [conf]
    """
    from pathlib import Path
    import cv2

    labels_dir = Path(labels_dir).expanduser()
    images_dir = Path(images_dir).expanduser()

    # Index images by stem (filename without extension)
    img_index = {}
    for p in images_dir.rglob("*"):
        if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}:
            img_index[p.stem] = p

    images = []
    annotations = []
    ann_id = 1
    img_id = 1
    num_boxes = 0

    for txt in sorted(labels_dir.glob("*.txt")):
        stem = txt.stem
        img_path = img_index.get(stem)
        if img_path is None:
            continue

        im = cv2.imread(str(img_path))
        if im is None:
            continue
        h, w = im.shape[:2]

        images.append(
            {
                "id": img_id,
                "file_name": f"images/test/{img_path.name}",
                "width": int(w),
                "height": int(h),
            }
        )

        with txt.open("r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue

                cls = int(float(parts[0]))
                xc, yc, bw, bh = map(float, parts[1:5])
                conf = float(parts[5]) if (has_conf and len(parts) >= 6) else 1.0

                if is_xywh_normalized:
                    xc *= w
                    yc *= h
                    bw *= w
                    bh *= h

                x = xc - bw / 2.0
                y = yc - bh / 2.0

                coco_cat = cls  # Assuming 0-based indexing

                annotations.append(
                    {
                        "id": ann_id,
                        "image_id": img_id,
                        "category_id": int(coco_cat),
                        "bbox": [float(x), float(y), float(bw), float(bh)],
                        "score": float(conf),
                    }
                )
                ann_id += 1
                num_boxes += 1

        img_id += 1

    return write_coco_output(
        images_folder=str(images_dir),
        model_name=model_name,
        categories_list=categories_list,
        images=images,
        annotations=annotations,
        num_images=len(images),
        num_boxes=num_boxes,
        total_train_time=float(total_train_time),
        total_inf_time=float(total_inf_time),
        output_path=str(output_path),
    )


In [20]:
labels_dir = (
    Path("~/Thesis_WDV/Yolo/yolo_outputs")
    .expanduser()
    / "apples_test_yolo11n_inst500"
    / "labels"
)
images_dir = Path("~/Thesis_WDV/Data/apples/images/test").expanduser()
out_dir = Path("~/Thesis_WDV/Results/Experiment_1").expanduser()

categories_list=["GoodApple","BadApple"]
total_train_time = 0.0
total_inf_time = 0.0

yolo_pred_txt_to_coco_results(
    labels_dir=labels_dir,
    images_dir=images_dir,
    output_path=out_dir,          # directory, no filename
    model_name="yolo11n_inst500",
    categories_list=categories_list,
    is_xywh_normalized=True,
    has_conf=True,
    total_train_time=total_train_time,
    total_inf_time=total_inf_time,
)



'/home/warredv/Thesis_WDV/Results/Experiment_1/apples_test_yolo11n_inst500_predictions.json'

In [6]:
import ultralytics
print(ultralytics.__version__)



8.3.239
